# Jaringan saraf rekuren

Dalam modul sebelumnya, kami membahas representasi semantik yang kaya dari teks. Arsitektur yang telah kami gunakan menangkap makna agregat dari kata-kata dalam sebuah kalimat, tetapi tidak mempertimbangkan **urutan** kata-kata tersebut, karena operasi agregasi yang mengikuti embedding menghilangkan informasi ini dari teks asli. Karena model-model ini tidak mampu merepresentasikan urutan kata, mereka tidak dapat menyelesaikan tugas yang lebih kompleks atau ambigu seperti pembuatan teks atau menjawab pertanyaan.

Untuk menangkap makna dari urutan teks, kita akan menggunakan arsitektur jaringan saraf yang disebut **jaringan saraf rekuren**, atau RNN. Saat menggunakan RNN, kita melewatkan kalimat kita ke jaringan satu token pada satu waktu, dan jaringan menghasilkan beberapa **status**, yang kemudian kita berikan lagi ke jaringan bersama token berikutnya.

![Gambar yang menunjukkan contoh pembuatan jaringan saraf rekuren.](../../../../../translated_images/id/rnn.27f5c29c53d727b5.webp)

Diberikan urutan input token $X_0,\dots,X_n$, RNN membuat urutan blok jaringan saraf, dan melatih urutan ini secara menyeluruh menggunakan backpropagation. Setiap blok jaringan mengambil pasangan $(X_i,S_i)$ sebagai input, dan menghasilkan $S_{i+1}$ sebagai hasil. Status akhir $S_n$ atau keluaran $Y_n$ masuk ke dalam pengklasifikasi linier untuk menghasilkan hasil. Semua blok jaringan berbagi bobot yang sama, dan dilatih secara menyeluruh menggunakan satu kali back propagation.

> Gambar di atas menunjukkan jaringan saraf rekuren dalam bentuk dibuka (di kiri), dan dalam representasi rekuren yang lebih ringkas (di kanan). Penting untuk menyadari bahwa semua Sel RNN memiliki **bobot yang dapat dibagikan** yang sama.

Karena vektor status $S_0,\dots,S_n$ diteruskan melalui jaringan, RNN mampu mempelajari ketergantungan berurutan antar kata. Misalnya, saat kata *not* muncul di suatu tempat dalam urutan, ia dapat belajar untuk meniadakan elemen tertentu dalam vektor status.

Di dalamnya, setiap sel RNN berisi dua matriks bobot: $W_H$ dan $W_I$, dan bias $b$. Pada setiap langkah RNN, diberikan input $X_i$ dan status input $S_i$, status keluaran dihitung sebagai $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, di mana $f$ adalah fungsi aktivasi (sering $\tanh$).

> Untuk masalah seperti pembuatan teks (yang akan kita bahas di unit berikutnya) atau terjemahan mesin, kita juga ingin mendapatkan nilai keluaran di setiap langkah RNN. Dalam kasus ini, ada juga matriks lain $W_O$, dan keluaran dihitung sebagai $Y_i=f(W_O\times S_i+b_O)$.

Mari kita lihat bagaimana jaringan saraf rekuren dapat membantu kita mengklasifikasikan dataset berita kita.

> Untuk lingkungan sandbox, kita perlu menjalankan sel berikut untuk memastikan pustaka yang dibutuhkan sudah terpasang, dan data sudah diambil terlebih dahulu. Jika Anda menjalankan secara lokal, Anda bisa melewati sel berikut.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Saat melatih model besar, alokasi memori GPU bisa menjadi masalah. Kita juga mungkin perlu bereksperimen dengan berbagai ukuran minibatch, agar data muat ke dalam memori GPU kita, namun pelatihannya cukup cepat. Jika Anda menjalankan kode ini di mesin GPU Anda sendiri, Anda bisa bereksperimen dengan mengatur ukuran minibatch untuk mempercepat pelatihan.

> **Catatan**: Versi driver NVidia tertentu diketahui tidak melepaskan memori setelah melatih model. Kami menjalankan beberapa contoh dalam notebook ini, dan hal itu dapat menyebabkan memori habis pada pengaturan tertentu, terutama jika Anda melakukan eksperimen sendiri dalam notebook yang sama. Jika Anda mengalami beberapa kesalahan aneh saat mulai melatih model, Anda mungkin ingin memulai ulang kernel notebook.


In [3]:
batch_size = 16
embed_size = 64

## Klasifikator RNN Sederhana

Dalam kasus RNN sederhana, setiap unit rekuren adalah jaringan linier sederhana, yang menerima vektor masukan dan vektor status, dan menghasilkan vektor status baru. Dalam Keras, ini dapat direpresentasikan oleh layer `SimpleRNN`.

Meskipun kita dapat langsung melewatkan token yang di-encode one-hot ke layer RNN, ini bukan ide yang baik karena dimensi mereka yang tinggi. Oleh karena itu, kita akan menggunakan layer embedding untuk menurunkan dimensi vektor kata, diikuti oleh layer RNN, dan akhirnya klasifikator `Dense`.

> **Catatan**: Dalam kasus di mana dimensinya tidak terlalu tinggi, misalnya saat menggunakan tokenisasi tingkat karakter, mungkin masuk akal untuk langsung melewatkan token yang di-encode one-hot ke dalam sel RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Catatan:** Kami menggunakan lapisan embedding yang belum dilatih di sini untuk kesederhanaan, tetapi untuk hasil yang lebih baik kami dapat menggunakan lapisan embedding yang sudah dilatih sebelumnya menggunakan Word2Vec, seperti yang dijelaskan di unit sebelumnya. Ini akan menjadi latihan yang bagus bagi Anda untuk menyesuaikan kode ini agar bekerja dengan embedding yang sudah dilatih sebelumnya.

Sekarang mari kita latih RNN kita. RNN secara umum cukup sulit dilatih, karena setelah sel RNN dibuka sepanjang panjang urutan, jumlah lapisan yang terlibat dalam backpropagation menjadi cukup besar. Oleh karena itu kita perlu memilih nilai learning rate yang lebih kecil, dan melatih jaringan pada dataset yang lebih besar untuk menghasilkan hasil yang baik. Ini bisa memakan waktu cukup lama, jadi menggunakan GPU lebih disarankan.

Untuk mempercepat proses, kita hanya akan melatih model RNN pada judul berita, mengabaikan deskripsi. Anda bisa mencoba melatih dengan deskripsi dan lihat apakah Anda dapat melatih modelnya.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Catatan** bahwa akurasi kemungkinan akan lebih rendah di sini, karena kami hanya melatih pada judul berita.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Sekarang kita menggunakan masking, kita dapat melatih model pada seluruh dataset judul dan deskripsi.

> **Catatan**: Apakah Anda memperhatikan bahwa kita telah menggunakan vectorizer yang dilatih pada judul berita, bukan seluruh isi artikel? Kemungkinan, ini dapat menyebabkan beberapa token diabaikan, jadi lebih baik melatih ulang vectorizer. Namun, ini mungkin hanya memiliki efek yang sangat kecil, jadi kita akan tetap menggunakan vectorizer pra-latih sebelumnya demi kesederhanaan.


## LSTM: Memori jangka pendek-panjang

Salah satu masalah utama dari RNN adalah **gradasi yang menghilang**. RNN bisa sangat panjang, dan mungkin mengalami kesulitan dalam menyebarkan gradasi hingga ke lapisan pertama dari jaringan selama backpropagation. Ketika ini terjadi, jaringan tidak dapat mempelajari hubungan antar token yang berjauhan. Salah satu cara untuk menghindari masalah ini adalah dengan memperkenalkan **manajemen status eksplisit** dengan menggunakan **gerbang**. Dua arsitektur paling umum yang mengenalkan gerbang adalah **memori jangka pendek-panjang** (LSTM) dan **unit relay yang digerakkan oleh gerbang** (GRU). Di sini kita akan membahas LSTM.

![Gambar yang menunjukkan contoh sel memori jangka pendek-panjang](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Jaringan LSTM diorganisir dengan cara yang mirip dengan RNN, tetapi ada dua status yang diteruskan dari lapisan ke lapisan: status aktual $c$, dan vektor tersembunyi $h$. Pada setiap unit, vektor tersembunyi $h_{t-1}$ digabungkan dengan input $x_t$, dan bersama-sama mereka mengendalikan apa yang terjadi pada status $c_t$ dan keluaran $h_{t}$ melalui **gerbang**. Setiap gerbang memiliki aktivasi sigmoid (keluaran dalam rentang $[0,1]$), yang dapat dianggap sebagai masker bitwise ketika dikalikan dengan vektor status. LSTM memiliki gerbang-gerbang berikut (dari kiri ke kanan pada gambar di atas):
* **gerbang lupa** yang menentukan komponen vektor $c_{t-1}$ yang harus kita lupakan, dan yang harus diteruskan. 
* **gerbang input** yang menentukan seberapa banyak informasi dari vektor input dan vektor tersembunyi sebelumnya harus dimasukkan ke dalam vektor status.
* **gerbang keluaran** yang mengambil vektor status baru dan memutuskan komponen mana yang akan digunakan untuk menghasilkan vektor tersembunyi baru $h_t$.

Komponen status $c$ dapat dianggap sebagai bendera yang dapat dihidupkan dan dimatikan. Misalnya, ketika kita menemukan nama *Alice* dalam urutan, kita memperkirakan bahwa itu merujuk pada seorang wanita, dan menaikkan bendera dalam status yang menyatakan kita memiliki kata benda feminin dalam kalimat. Ketika kita kemudian menemukan kata *and Tom*, kita akan menaikkan bendera yang menyatakan kita memiliki kata benda jamak. Dengan demikian dengan memanipulasi status kita dapat melacak properti tata bahasa dari kalimat tersebut.

> **Catatan**: Berikut adalah sumber yang bagus untuk memahami bagian internal LSTM: [Memahami Jaringan LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) oleh Christopher Olah.

Meskipun struktur internal dari sel LSTM mungkin terlihat rumit, Keras menyembunyikan implementasi ini di dalam lapisan `LSTM`, jadi satu-satunya hal yang perlu kita lakukan dalam contoh di atas adalah mengganti lapisan rekuren:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Perhatikan** bahwa melatih LSTM juga cukup lambat, dan Anda mungkin tidak melihat peningkatan akurasi yang signifikan pada awal pelatihan. Anda mungkin perlu melanjutkan pelatihan untuk beberapa waktu guna mencapai akurasi yang baik.


## RNN Dua Arah dan Multilayer

Dalam contoh kita sejauh ini, jaringan berulang beroperasi dari awal sebuah urutan hingga akhir. Ini terasa alami bagi kita karena mengikuti arah yang sama dengan cara kita membaca atau mendengarkan ucapan. Namun, untuk skenario yang memerlukan akses acak pada urutan input, lebih masuk akal untuk menjalankan perhitungan berulang dalam kedua arah. RNN yang memungkinkan perhitungan dalam kedua arah disebut RNN **dua arah**, dan dapat dibuat dengan membungkus lapisan berulang dengan lapisan `Bidirectonal` khusus.

> **Catatan**: Lapisan `Bidirectional` membuat dua salinan dari lapisan di dalamnya, dan menetapkan properti `go_backwards` dari salah satu salinan tersebut ke `True`, membuatnya berjalan ke arah berlawanan sepanjang urutan.

Jaringan berulang, baik unidirectional maupun bidirectional, menangkap pola dalam sebuah urutan, dan menyimpannya ke dalam vektor keadaan atau mengembalikannya sebagai keluaran. Sama seperti jaringan konvolusional, kita dapat membangun lapisan berulang lain setelah yang pertama untuk menangkap pola dengan level yang lebih tinggi, yang dibangun dari pola dengan level yang lebih rendah yang diekstraksi oleh lapisan pertama. Ini membawa kita pada konsep **RNN multi-layer**, yang terdiri dari dua atau lebih jaringan berulang, di mana keluaran dari lapisan sebelumnya diteruskan ke lapisan berikutnya sebagai input.

![Gambar yang menunjukkan RNN multilayer long-short-term-memory](../../../../../translated_images/id/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Gambar dari [postingan hebat ini](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) oleh Fernando López.*

Keras membuat konstruksi jaringan ini menjadi mudah, karena Anda hanya perlu menambahkan lebih banyak lapisan berulang ke model. Untuk semua lapisan kecuali yang terakhir, kita perlu menentukan parameter `return_sequences=True`, karena kita membutuhkan lapisan tersebut mengembalikan semua status perantara, bukan hanya status akhir dari perhitungan berulang.

Mari kita buat LSTM dua lapis bidirectional untuk masalah klasifikasi kita.

> **Catatan** kode ini juga membutuhkan waktu yang cukup lama untuk selesai, tetapi memberikan akurasi tertinggi yang pernah kita lihat sejauh ini. Jadi mungkin sebaiknya menunggu dan melihat hasilnya.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN untuk tugas lainnya

Sampai saat ini, kita telah fokus menggunakan RNN untuk mengklasifikasikan urutan teks. Namun mereka dapat menangani banyak tugas lainnya, seperti generasi teks dan terjemahan mesin &mdash; kita akan mempertimbangkan tugas-tugas tersebut di unit berikutnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
